In [ ]:
### Imports

import pathlib

import h5py
import matplotlib.pyplot as plt
import numpy as np
import polars as pls

import usv_playpen.analyses.neuronal_coactivity_engine as engine

In [ ]:
### Load the data

data_directory = pathlib.Path('/mnt/falkner/Bartul/Data/20250919_152924') # 20250923_162249 20250919_152924 20250923_200538

# Load tracking data
tracking_file = next(item for item in data_directory.glob('**/*_translated_rotated_metric.h5'))

with h5py.File(name=tracking_file, mode='r') as track_file:
    mouse_tracks = np.array(track_file['tracks'])
    mouse_track_names = [item.decode('utf-8') for item in list(track_file['track_names'])]
    recording_frame_rate = float(track_file['recording_frame_rate'][()])

# Load vocalization data and filter for male USVs
usv_summary_file = next(item for item in data_directory.glob('**/*_usv_summary.csv'))
usv_summary_data = pls.read_csv(usv_summary_file)
usv_summary_data = usv_summary_data.filter(pls.col('emitter') == mouse_track_names[0])

# Load spike data (in seconds, not tracking frames)
neural_data_dict = {}
for spike_file in data_directory.glob('**/*_good.npy'):
    neural_data_dict[spike_file.stem] = np.load(spike_file)[0, :]

In [ ]:
### Compute Coactivity Metrics for Complex vs. Simple Vocalizations

usv_bootstrap_num = 60    
n_boot_iterations = 1000
n_shuffles = 1000
WINDOW_S = 0.030 

# 1. Filter groups
complex_df = usv_summary_data.filter(pls.col('usv_supercategory').is_in([1, 2]))
simple_df = usv_summary_data.filter(pls.col('usv_supercategory') == 5)

total_duration = mouse_tracks.shape[0] / recording_frame_rate
rng = np.random.default_rng()

print(f"Total Vocalizations: {len(usv_summary_data)} | Complex: {len(complex_df)} | Simple: {len(simple_df)}")

# 2. Extract count matrices (neurons x trials)
complex_counts = engine.extract_snippet_matrix(
    onsets=complex_df['start'].to_numpy(), 
    neural_data=neural_data_dict, 
    window_s=WINDOW_S
)

simple_counts = engine.extract_snippet_matrix(
    onsets=simple_df['start'].to_numpy(), 
    neural_data=neural_data_dict, 
    window_s=WINDOW_S
)

# 3. Symmetric Bootstrapping (N=matched)
print(f"Bootstrapping both groups to matched N={usv_bootstrap_num}...")

complex_boot = engine.get_bootstrapped_simple_calls(
    simple_counts=complex_counts, 
    n_target=usv_bootstrap_num, 
    n_iterations=n_boot_iterations
)

simple_boot = engine.get_bootstrapped_simple_calls(
    simple_counts=simple_counts, 
    n_target=usv_bootstrap_num, 
    n_iterations=n_boot_iterations
)

# 4. Symmetric Circular Shuffle (N=matched)
print(f"Generating matched-N null distributions...")

# Sample timing templates to match N
complex_onsets_sub = rng.choice(complex_df['start'].to_numpy(), size=usv_bootstrap_num, replace=False)
simple_onsets_sub = rng.choice(simple_df['start'].to_numpy(), size=usv_bootstrap_num, replace=False)

complex_null_dist = engine.perform_circular_shuffle(
    onsets=np.sort(complex_onsets_sub), 
    neural_data=neural_data_dict, 
    total_duration=total_duration, 
    window_s=WINDOW_S, 
    n_shuffles=n_shuffles
)

simple_null_dist = engine.perform_circular_shuffle(
    onsets=np.sort(simple_onsets_sub), 
    neural_data=neural_data_dict, 
    total_duration=total_duration, 
    window_s=WINDOW_S, 
    n_shuffles=n_shuffles
)

# 5. Statistical Validation
def calculate_stats(boot_data, null_data):
    boot_mean = np.mean(boot_data)
    null_mean = np.mean(null_data)
    null_std = np.std(null_data)
    p_val = np.mean(null_data >= boot_mean)
    z_score = (boot_mean - null_mean) / null_std if null_std > 0 else 0
    return boot_mean, null_mean, p_val, z_score

# Compute for COMPLEX
c_rsc_obs, c_rsc_null, c_rsc_p, c_rsc_z = calculate_stats(complex_boot['r_sc'], complex_null_dist['r_sc'])
c_sim_obs, c_sim_null, c_sim_p, c_sim_z = calculate_stats(complex_boot['similarity'], complex_null_dist['similarity'])
c_pop_obs, c_pop_null, c_pop_p, c_pop_z = calculate_stats(complex_boot['pop_corr'], complex_null_dist['pop_corr'])

# Compute for SIMPLE
s_rsc_obs, s_rsc_null, s_rsc_p, s_rsc_z = calculate_stats(simple_boot['r_sc'], simple_null_dist['r_sc'])
s_sim_obs, s_sim_null, s_sim_p, s_sim_z = calculate_stats(simple_boot['similarity'], simple_null_dist['similarity'])
s_pop_obs, s_pop_null, s_pop_p, s_pop_z = calculate_stats(simple_boot['pop_corr'], simple_null_dist['pop_corr'])

# 6. Display Summary Results
print("-" * 75)
print(f"SYMMETRIC COMPARISON (Matched N={usv_bootstrap_num})")
print("-" * 75)

print(f"COMPLEX (Resampled N={usv_bootstrap_num}):")
print(f"  r_sc:       Boot={c_rsc_obs:.4f} | Null={c_rsc_null:.4f} | p={c_rsc_p:.4f} (Z={c_rsc_z:.2f})")
print(f"  Similarity: Boot={c_sim_obs:.4f} | Null={c_sim_null:.4f} | p={c_sim_p:.4f} (Z={c_sim_z:.2f})")
print(f"  Pop Corr:   Boot={c_pop_obs:.4f} | Null={c_pop_null:.4f} | p={c_pop_p:.4f} (Z={c_pop_z:.2f})")

print("-" * 75)

print(f"SIMPLE (Resampled N={usv_bootstrap_num}):")
print(f"  r_sc:       Boot={s_rsc_obs:.4f} | Null={s_rsc_null:.4f} | p={s_rsc_p:.4f} (Z={s_rsc_z:.2f})")
print(f"  Similarity: Boot={s_sim_obs:.4f} | Null={s_sim_null:.4f} | p={s_sim_p:.4f} (Z={s_sim_z:.2f})")
print(f"  Pop Corr:   Boot={s_pop_obs:.4f} | Null={s_pop_null:.4f} | p={s_pop_p:.4f} (Z={s_pop_z:.2f})")

In [ ]:

# 1. Calculate category-specific thresholds (99th Percentile)
# We handle all three metrics for the matched-N distributions
c_rsc_99 = np.percentile(complex_null_dist['r_sc'], 99)
c_sim_99 = np.percentile(complex_null_dist['similarity'], 99)
c_pop_99 = np.percentile(complex_null_dist['pop_corr'], 99)

s_rsc_99 = np.percentile(simple_null_dist['r_sc'], 99)
s_sim_99 = np.percentile(simple_null_dist['similarity'], 99)
s_pop_99 = np.percentile(simple_null_dist['pop_corr'], 99)

# 2. Extract Bootstrap Means (Resampled N=45)
c_val_rsc = np.mean(complex_boot['r_sc'])
c_val_sim = np.mean(complex_boot['similarity'])
c_val_pop = np.mean(complex_boot['pop_corr'])

s_val_rsc = np.mean(simple_boot['r_sc'])
s_val_sim = np.mean(simple_boot['similarity'])
s_val_pop = np.mean(simple_boot['pop_corr'])

# 3. Plotting: 3 Rows (Metric Type) x 2 Columns (Call Type)
fig, axes = plt.subplots(3, 2, figsize=(14, 15), sharey='row')

# --- ROW 1: Pairwise Correlation (r_sc) ---
# Complex
ax = axes[0, 0]
ax.hist(complex_null_dist['r_sc'], bins=40, color='grey', alpha=0.5, label='Null')
ax.axvline(c_rsc_99, color='black', linestyle='--', label='99%')
ax.axvline(c_val_rsc, color='crimson', linewidth=3, label=f'Boot ({c_val_rsc:.4f})')
ax.set_title('COMPLEX: Pairwise Correlation ($r_{sc}$)', fontsize=14)
ax.legend(fontsize=8)

# Simple
ax = axes[0, 1]
ax.hist(simple_null_dist['r_sc'], bins=40, color='grey', alpha=0.5, label='Null')
ax.axvline(s_rsc_99, color='black', linestyle='--', label='99%')
ax.axvline(s_val_rsc, color='dodgerblue', linewidth=3, label=f'Boot ({s_val_rsc:.4f})')
ax.set_title('SIMPLE: Pairwise Correlation ($r_{sc}$)', fontsize=14)
ax.legend(fontsize=8)

# --- ROW 2: Population Similarity (Cosine) ---
# Complex
ax = axes[1, 0]
ax.hist(complex_null_dist['similarity'], bins=40, color='grey', alpha=0.5, label='Null')
ax.axvline(c_sim_99, color='black', linestyle='--', label='99%')
ax.axvline(c_val_sim, color='crimson', linewidth=3, label=f'Boot ({c_val_sim:.4f})')
ax.set_title('COMPLEX: Cosine Similarity', fontsize=14)
ax.legend(fontsize=8)

# Simple
ax = axes[1, 1]
ax.hist(simple_null_dist['similarity'], bins=40, color='grey', alpha=0.5, label='Null')
ax.axvline(s_sim_99, color='black', linestyle='--', label='99%')
ax.axvline(s_val_sim, color='dodgerblue', linewidth=3, label=f'Boot ({s_val_sim:.4f})')
ax.set_title('SIMPLE: Cosine Similarity', fontsize=14)
ax.legend(fontsize=8)

# --- ROW 3: Population Vector Correlation (Pearson) ---
# Complex
ax = axes[2, 0]
ax.hist(complex_null_dist['pop_corr'], bins=40, color='grey', alpha=0.5, label='Null')
ax.axvline(c_pop_99, color='black', linestyle='--', label='99%')
ax.axvline(c_val_pop, color='crimson', linewidth=3, label=f'Boot ({c_val_pop:.4f})')
ax.set_title('COMPLEX: Pop Vector Corr', fontsize=14)
ax.set_xlabel('Mean Pearson Correlation', fontsize=12)
ax.legend(fontsize=8)

# Simple
ax = axes[2, 1]
ax.hist(simple_null_dist['pop_corr'], bins=40, color='grey', alpha=0.5, label='Null')
ax.axvline(s_pop_99, color='black', linestyle='--', label='99%')
ax.axvline(s_val_pop, color='dodgerblue', linewidth=3, label=f'Boot ({s_val_pop:.4f})')
ax.set_title('SIMPLE: Pop Vector Corr', fontsize=14)
ax.set_xlabel('Mean Pearson Correlation', fontsize=12)
ax.legend(fontsize=8)

# Cleanup spines and layout
for ax_row in axes:
    for ax in ax_row:
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()